# IgniteCyber 5‑Day Cyber AI Analyst Bootcamp — LAB3.3

**Offline-only** • **Sigma-first** • **Elastic/Kibana + TheHive 5 + Cortex + MISP**  
**Datasets:** `/opt/bootcamp/datasets` • **Notebooks:** `/opt/bootcamp/notebooks`

## CJ Intel Spine (Single evolving MISP Event + one Event Report)
Throughout the week, we maintain **one** MISP event for the threat actor **Cinder Jackal (CJ)** and append a **single MISP Event Report** day-by-day.

**Workflow (Notebook-guided / manual in TheHive UI):**
1. Hunt/confirm in **Kibana** (`zeek-*`, `wazuh-alerts-*`, `elastic-agent-*`)
2. Add observables + evidence to **TheHive case**
3. Run **Cortex analyzers** from TheHive (sightings / file triage)
4. Promote validated intel into **MISP objects**
5. Append the narrative block generated by this notebook into the **CJ MISP Event Report**

## Required dataset bundles for this lab
- (see course dataset catalog for this lab bundle)


# Lab 3.3 — Timeline Reconstruction (Elastic-based, Timesketch optional)


## Scenario storyline (persistent across the week)

**Company:** ApexFin Solutions (AF) — a fast-growing fintech startup that provides a consumer budgeting app and small-business payment rails.

**Business critical assets:**
- AF-PAY API (customer payments)
- AF-PORTAL (employee/admin portal)
- AF-ANALYTICS (data warehouse exports)

**Threat actor:** CINDER JACKAL (fictional) — financially motivated intrusion set that blends commodity tradecraft with “living-off-the-land” execution and quick monetization.

**CINDER JACKAL objectives:**
1) Obtain employee credentials via phishing and browser session theft  
2) Establish persistence and stealthy C2  
3) Identify and exfiltrate customer PII and payment reconciliation data  
4) Extort FinanceFront with data-theft + disruption pressure

We use **MITRE ATT&CK** to describe behavior, **Sigma-first detections** for portability, and **Elastic/Kibana + Wazuh/Zeek** telemetry for investigations.



### Lab VM / Data assumptions (offline-only)

- Datasets (ZIP) are available locally in: `/opt/bootcamp/datasets`
- Elastic/Kibana index patterns:
  - `zeek-*`
  - `wazuh-alerts-*`
  - `elastic-agent-*`
- Elastic user/pass are preconfigured in the VM (read from environment variables by default).
- Local LLM: **Ollama** on `http://localhost:11434` (no internet required).


## Objectives
- Complete the tasks for **Lab 3.3** and capture evidence.
- Map observed behavior to MITRE ATT&CK.
- Produce a Sigma-first detection outcome.

## MITRE ATT&CK mapping
- **TA0009 / T1074.001** — Local Data Staging

## Datasets (offline ZIPs)
_(No external dataset required — use the pre-indexed lab telemetry.)_

## Tasks (do these in order)
1. Create a time-ordered sequence of key events across endpoint + network data.
1. Mark phase boundaries: initial access → execution → persistence → C2 → actions on objectives.
1. Export a timeline evidence CSV and write a 1-paragraph narrative.


> Attribution / inspiration: see `docs/references.md` (Security Break Jupyter Collection + Security Datasets).


In [ ]:
import os
from bootcamp_lib import *
print('ES_HOST:', ES_HOST)
print('Ollama models:', ollama_models()[:5])
print('Dataset ZIP count:', len(list_dataset_zips()))


## Step 1 — Elastic health checks


In [ ]:
info = es_get('/')
print(json.dumps({k: info.get(k) for k in ['name','cluster_name','cluster_uuid','version','tagline']}, indent=2))
health = es_get('/_cluster/health')
print(json.dumps({k: health.get(k) for k in ['status','number_of_nodes','active_primary_shards','active_shards','unassigned_shards']}, indent=2))


## Step 2 — Confirm index patterns exist (zeek, wazuh, elastic-agent)


In [ ]:
aliases = es_get('/_aliases')
idx = sorted(list(aliases.keys()))
patterns = ['zeek-','wazuh-alerts-','elastic-agent-']
found = {p:[i for i in idx if i.startswith(p)] for p in patterns}
for p, items in found.items():
    print(p, '=>', len(items), 'indices')
    for i in items[:10]:
        print('  -', i)


## Step 3 — Pull a broad, time-ordered event set from Elastic

Start wide, then narrow. The goal here is to build a candidate event stream across **endpoint + network** sources before you decide what truly belongs in the incident timeline.

### Focus areas
- Endpoint process activity (`process.name`, `process.command_line`, `host.name`, `user.name`)
- File/path evidence that may suggest **staging** (`file.path`, temp paths, archive tools, copy tools)
- Network follow-on activity from Zeek (`zeek.conn`, `zeek.http`, `zeek.dns`)
- Any event message or action that helps place activity in sequence

### Investigation goal
Create a rough timeline seed that you can sort by `@timestamp` and then refine in the next steps.


In [ ]:
import pandas as pd

timeline_qs = '(@timestamp>=now-7d) AND (event.category:(process OR network OR file) OR event.dataset:(zeek.conn OR zeek.http OR zeek.dns) OR process.name:* OR file.path:*)'
res = es_qs('elastic-agent-*,wazuh-alerts-*,zeek-*', timeline_qs, size=1000)
raw_df = hits_to_df(res)

print('Events returned:', len(raw_df))

interesting_cols = [
    '@timestamp', 'event.dataset', 'event.category', 'event.action',
    'host.name', 'user.name',
    'process.name', 'process.parent.name', 'process.command_line',
    'file.path', 'source.ip', 'destination.ip',
    'dns.question.name', 'url.full', 'message'
]

timeline_seed = raw_df[[c for c in interesting_cols if c in raw_df.columns]].copy() if len(raw_df) else pd.DataFrame(columns=interesting_cols)

if '@timestamp' in timeline_seed.columns:
    timeline_seed['@timestamp'] = pd.to_datetime(timeline_seed['@timestamp'], errors='coerce', utc=True)
    timeline_seed = timeline_seed.sort_values('@timestamp').reset_index(drop=True)

display(timeline_seed.head(25))
print('Columns available for timeline work:')
print(list(timeline_seed.columns))


## Step 4 — Normalize the timeline view and guess phase labels

Now convert the raw event set into a compact timeline table that is easier to review.

### What to do
- Keep the most useful columns for investigation
- Sort strictly by time
- Add a **best-effort phase guess** so you can quickly spot:
  - `initial_access`
  - `execution`
  - `persistence`
  - `c2`
  - `actions_on_objectives`
  - `review`

> Phase labels are only a starting point. You still need to verify each one manually in Kibana.


In [ ]:
import pandas as pd

def _col(df, name, default=''):
    return df[name] if name in df.columns else pd.Series([default] * len(df))

def _guess_phase(row):
    text = ' '.join([
        str(row.get('dataset', '')),
        str(row.get('process', '')),
        str(row.get('parent', '')),
        str(row.get('command_line', '')),
        str(row.get('file_path', '')),
        str(row.get('message', '')),
        str(row.get('url', '')),
        str(row.get('dns', '')),
    ]).lower()

    if any(x in text for x in ['phish', 'outlook', 'winword', 'excel', 'attachment', 'url', 'macro']):
        return 'initial_access'
    if any(x in text for x in ['powershell', 'cmd.exe', 'mshta', 'rundll32', 'wscript', 'cscript', 'regsvr32']):
        return 'execution'
    if any(x in text for x in ['schtasks', 'run key', 'runkeys', 'startup', 'service install', 'registry run']):
        return 'persistence'
    if any(x in text for x in ['zeek.conn', 'beacon', 'http', 'dns', 'tls', 'winrm', 'c2']):
        return 'c2'
    if any(x in text for x in ['7z', 'rar', 'zip', 'compress-archive', 'makecab', 'tar', 'copy', 'xcopy', 'robocopy', '\\temp\\', '\\users\\public\\', 'staging', 'exfil']):
        return 'actions_on_objectives'
    return 'review'

if len(timeline_seed):
    timeline_df = pd.DataFrame({
        'timestamp': _col(timeline_seed, '@timestamp'),
        'dataset': _col(timeline_seed, 'event.dataset'),
        'category': _col(timeline_seed, 'event.category'),
        'action': _col(timeline_seed, 'event.action'),
        'host': _col(timeline_seed, 'host.name'),
        'user': _col(timeline_seed, 'user.name'),
        'process': _col(timeline_seed, 'process.name'),
        'parent': _col(timeline_seed, 'process.parent.name'),
        'command_line': _col(timeline_seed, 'process.command_line'),
        'file_path': _col(timeline_seed, 'file.path'),
        'source_ip': _col(timeline_seed, 'source.ip'),
        'destination_ip': _col(timeline_seed, 'destination.ip'),
        'dns': _col(timeline_seed, 'dns.question.name'),
        'url': _col(timeline_seed, 'url.full'),
        'message': _col(timeline_seed, 'message'),
    }).sort_values('timestamp').reset_index(drop=True)

    timeline_df['phase_guess'] = timeline_df.apply(_guess_phase, axis=1)
    display(timeline_df.head(40))
    print('Phase guesses:')
    print(timeline_df['phase_guess'].value_counts(dropna=False))
else:
    timeline_df = pd.DataFrame()
    print('timeline_seed is empty. Broaden the Step 3 query or increase size.')


## Step 5 — Isolate likely staging / archive activity and mark phase boundaries

This lab is mapped to **T1074.001 — Local Data Staging**, so you want to look for evidence that files were being collected, copied, compressed, or prepared before movement or exfiltration.

### What to look for
- Archive/compression utilities: `7z`, `rar`, `zip`, `tar`, `makecab`, `Compress-Archive`
- Copy/move behavior: `copy`, `xcopy`, `robocopy`, temp/public staging paths
- Follow-on network activity shortly after staging
- Any process + file path combination that helps you define:
  - first suspicious execution
  - first persistence signal
  - first C2/network callback
  - first staging / objective action


In [ ]:
staging_qs = 'process.command_line:(*7z* OR *rar* OR *zip* OR *Compress-Archive* OR *makecab* OR *tar* OR *robocopy* OR *xcopy* OR *copy *) OR file.path:(*\\Users\\Public\\* OR *\\AppData\\Local\\Temp\\* OR *\\ProgramData\\* OR *\\Temp\\*)'
res_stage = es_qs('elastic-agent-*,wazuh-alerts-*', staging_qs, size=300)
stage_df = hits_to_df(res_stage)

print('Potential staging-related events:', len(stage_df))
display(stage_df.head(25))

phase_boundaries = {
    'initial_access_start': '',
    'execution_start': '',
    'persistence_start': '',
    'c2_start': '',
    'actions_on_objectives_start': '',
}

timeline_notes = {
    'suspicious_host': '',
    'suspicious_user': '',
    'archive_or_copy_commands': [],
    'staging_paths': [],
    'follow_on_network_activity': [],
}

print('Fill phase_boundaries and timeline_notes as you validate findings in Kibana:')
print(json.dumps(phase_boundaries, indent=2))
print(json.dumps(timeline_notes, indent=2))


## Step 6 — Starter investigation queries (copy/paste into Kibana)

Use these to refine the final timeline and verify phase transitions:

- `@timestamp>=now-7d and (process.name:* or event.dataset:(zeek.conn OR zeek.http OR zeek.dns))`
- `process.command_line:(*7z* OR *rar* OR *zip* OR *Compress-Archive* OR *makecab* OR *tar*)`
- `process.command_line:(*copy * OR *xcopy* OR *robocopy*)`
- `file.path:(*\\Users\\Public\\* OR *\\AppData\\Local\\Temp\\* OR *\\ProgramData\\* OR *\\Temp\\*)`
- `host.name:"<host>" and @timestamp:[<start> TO <end>]`
- `host.name:"<host>" and process.command_line:(*powershell* OR *cmd.exe* OR *schtasks* OR *reg add* OR *rundll32*)`
- `event.dataset:zeek.conn and destination.ip:* and @timestamp:[<start> TO <end>]`
- `event.dataset:zeek.dns and dns.question.name:* and host.name:"<host>"`

### Analyst task
Before moving to Step 7, decide:
1. Which **host** is your primary timeline host
2. Which **time window** best represents the intrusion
3. Which **3–10 events** are essential for your final narrative


In [ ]:
starter_queries = [
    '@timestamp>=now-7d and (process.name:* or event.dataset:(zeek.conn OR zeek.http OR zeek.dns))',
    'process.command_line:(*7z* OR *rar* OR *zip* OR *Compress-Archive* OR *makecab* OR *tar*)',
    'process.command_line:(*copy * OR *xcopy* OR *robocopy*)',
    'file.path:(*\\Users\\Public\\* OR *\\AppData\\Local\\Temp\\* OR *\\ProgramData\\* OR *\\Temp\\*)',
    'host.name:"<host>" and @timestamp:[<start> TO <end>]',
    'host.name:"<host>" and process.command_line:(*powershell* OR *cmd.exe* OR *schtasks* OR *reg add* OR *rundll32*)',
    'event.dataset:zeek.conn and destination.ip:* and @timestamp:[<start> TO <end>]',
    'event.dataset:zeek.dns and dns.question.name:* and host.name:"<host>"',
]
print('\n'.join(starter_queries))


## Step 7 — Build an evidence packet (for reporting + AI assistance)
Adjust `sample_qs` to match what you found in Steps 3–6.


In [ ]:
sample_qs = 'process.command_line:(*7z* OR *rar* OR *zip* OR *Compress-Archive* OR *makecab* OR *tar* OR *robocopy* OR *xcopy*) OR file.path:(*\\Users\\Public\\* OR *\\AppData\\Local\\Temp\\* OR *\\ProgramData\\* OR *\\Temp\\*) OR event.dataset:(zeek.conn OR zeek.http OR zeek.dns)'
# Refine this query to the host, time window, and timeline events you confirmed in Steps 3–6.
res = es_qs('elastic-agent-*,wazuh-alerts-*,zeek-*', sample_qs, size=500)
df = hits_to_df(res)
out_path = save_evidence_csv(df, 'Lab_3.3')
print('Saved evidence CSV:', out_path, 'rows:', len(df))
display(df.head(15))


## Step 8 — AI assistance (local Ollama) + human verification
**AI task for this lab:** Ask Ollama to convert the timeline to an executive summary, but require evidence citations for each claim.

Rule: **No evidence → no claims.** Validate output with additional queries.


In [ ]:
models = ollama_models()
model = models[0] if models else "llama3"
evidence_preview = df.head(30).to_csv(index=False) if 'df' in globals() and isinstance(df, pd.DataFrame) else ""
prompt = f'''You are a SOC threat hunter. Work offline. Use only the evidence provided.

OUTPUT FORMAT (strict):
- Summary (5 bullets)
- ATT&CK techniques (ID + 1-line justification referencing evidence columns)
- Validation queries (3 KQL queries)
- Containment actions (2)
- Uncertainty (what is missing / what could be wrong)

EVIDENCE (CSV excerpt):
{evidence_preview}
'''
if evidence_preview.strip():
    response = ollama_generate(prompt, model=model, temperature=0.2)
    print(response)
else:
    print("No evidence dataframe found yet. Run Step 7 first.")


## Deliverables / Checkpoint
- Evidence CSV path (saved in `/tmp/`)
- ATT&CK technique IDs you verified
- Sigma rule or tuning notes (if applicable)
- 1-paragraph narrative (what happened + why you believe it)


## CJ Event Report (copy/paste into MISP → Event → Event Reports)

**Report name (single for the week):** `Cinder Jackal — Weekly Narrative (Bootcamp)`  
In MISP: open the **CJ Event** → **Event Reports** → open the weekly report → paste the block below at the end.

> Tip: Keep headings consistent so the report reads like a timeline.

### Paste this block


In [ ]:
from datetime import datetime
import os, textwrap

LAB_CODE = "LAB3.3"
DAY_LABEL = os.getenv("BOOTCAMP_DAY", "") or ""
TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%SZ")

# Fill these as you work the lab (concise + evidence-driven)
findings = {
    "summary": "",
    "what_happened": [],
    "key_iocs": [],
    "attack_mapping": [],
    "elastic_evidence": [],
    "thehive_actions": [],
    "misp_updates": [],
    "next_steps": [],
}

def _lines(items, fallback="(fill in)"):
    return "\n".join(["- " + x for x in (items if items else [fallback])])

def render_report_block():
    md = f"""\
## {DAY_LABEL + " — " if DAY_LABEL else ""}{LAB_CODE} — CJ Narrative Update
**Timestamp (UTC):** {TS}

**Summary:** {findings["summary"] or "(fill in)"}

**What happened**
{_lines(findings["what_happened"])}

**Key IOCs**
{_lines(findings["key_iocs"])}

**MITRE ATT&CK mapping**
{_lines(findings["attack_mapping"])}

**Elastic evidence (queries / fields / indices)**
{_lines(findings["elastic_evidence"])}

**TheHive actions**
{_lines(findings["thehive_actions"])}

**MISP updates (objects + tags)**
{_lines(findings["misp_updates"])}

**Next steps**
{_lines(findings["next_steps"])}

---"""
    return textwrap.dedent(md)

report_block = render_report_block()
print(report_block)

out_dir = "/opt/bootcamp/reports/cj"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f"{LAB_CODE.replace('.','_')}_cj_event_report_update.md")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(report_block)

print(f"Saved: {out_path}")
